# 01 · Convert a detector: SBND from its GDML

lumen does not read GDML (the geometry file Geant4 reads) while it simulates. It reads a **detector file**: the
geometry, materials, optical surfaces and light sensors, converted once and checked. This notebook makes SBND's.

A GDML file says what Geant4 builds, but not everything lumen needs: which volume is the detector, which volumes
are sensors and how they read light. That is the **recipe**, which lumen-tpc keeps for each LArSoft detector
(`lumen_tpc.larsoft`), with its **inputs**: for SBND, sbndcode's map of which optical channel is which kind of
sensor.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import lumen
import lumen_tpc
from lumen.compare import references

WORK = Path(os.environ.get("LUMEN_NOTEBOOKS", "lumen-notebooks"))
WORK.mkdir(parents=True, exist_ok=True)

entry = lumen_tpc.DETECTORS["sbnd"]  # a lumen.DetectorEntry
recipe, gdml, inputs = entry.recipe, entry.gdml, entry.inputs
print("GDML:", gdml, " inputs:", inputs)
print("detector volume:", recipe.detector_volume)
for g in recipe.groups:
    print(f"sensor group {g.name!r}: counted by rule {g.rule!r}, read with value set {g.values!r}")

## Convert

`lumen.convert("lumen_tpc:sbnd", ...)` reads that entry of `lumen_tpc.DETECTORS` (a `lumen.DetectorEntry`: the recipe, the GDML and the
inputs, under the reference data directory `references.store()`: `LUMEN_DATA`, else lumen's cache; a GDML it lacks
is fetched and checked first) and converts it. On the way lumen applies its
general import rules and the recipe's decisions (for SBND, the cathode's reflector foil becomes a wavelength-shifting
TPB half and a reflector half: TPB is a volume lumen navigates like any other), checks the structure, and runs the
**placement check**: Geant4 assumes that volumes placed side by side never overlap and that a volume lies inside its
mother; lumen tests both on points just inside every surface (what Geant4's `CheckOverlaps` does) and records what
it finds. It never changes the geometry.

In [ ]:
path = WORK / "sbnd.lumen"
t0 = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    lumen.convert("lumen_tpc:sbnd", out=path, data=references.store(), overwrite=True)
print(f"converted in {time.perf_counter() - t0:.0f} s to {path}")
for w in caught:
    if "placement" in w.category.__name__.lower():
        print("placement warning:", str(w.message)[:400])

## The geometry and overlap checks

The conversion's report lists every decision and the placement check's verdict. SBND's GDML has overlapping
volumes: the field cage pokes into the gas above the argon, and field-cage strips touch their support structure.
Geant4 would navigate such places in an undefined way; lumen gives each shared region one owner (named in the
report) and says so. `lumen.compare.ownership.export` writes a GDML in which Geant4 sees the same ownership, for a
like-for-like comparison: the Geant4 references the family's checks pass or fail against ran that export.

In [ ]:
det = lumen.load(path)
for line in det.file.report:
    print("-", line)

The **geometry check** goes further: it asks Geant4 itself where random points are (inside which volume, which
material) and compares with lumen's answer, point by point. It needs a Geant4 build with the `g4ref` probe, which
you point to with the environment variable `G4REF`; without it this cell only says so. On SBND it does not pass:
a fraction of a percent of the points are placed differently by Geant4 and by lumen. Where the GDML's volumes
overlap (the report above) each navigator picks its own owner, so some disagreement is expected there; whether it
accounts for every mismatched point is not yet checked.

In [ ]:
g4ref = os.environ.get("G4REF")
if g4ref:
    from lumen.compare import geometry

    rows = geometry.validate("sbnd", detector=path, g4ref=g4ref, extra=[lumen_tpc.REFERENCES])
    for r in rows:
        print(r["quantity"], r["value"], "pass" if r["pass"] else "FAIL", r["extra"])
else:
    print("G4REF is not set: run  python -m lumen.compare geometry sbnd --package lumen_tpc --detector-file",
          path, "--g4ref BINARY")

## What the file holds

A detector file is a directory of plain JSON and arrays. The loaded `Detector` summarises it:

In [ ]:
print(det)
print("files:", sorted(p.name for p in path.iterdir()))
print(len(det.file.materials), "materials, for example", list(det.file.materials)[:8])
print("optical surfaces:", list(det.file.surfaces))
print("converted from:", det.file.provenance["source"])

Every number lumen uses, except the geometry, is a **parameter** with a name, a unit, an allowed range and a
*gradient method* (how lumen differentiates it, notebook 03). `explain` says where the value came from.

In [ ]:
p0 = det.params()
print(len(p0), "parameters, for example:")
for key in ("materials/LAr/RAYLEIGH", "materials/LAr/SCINTILLATIONYIELD", "materials/LAr/SCINTILLATIONTIMECONSTANT2",
            "surfaces/Surface_vm2000/REFLECTIVITY"):
    info = p0.info(key)
    print(f"  {key:42s} unit {info.unit or '-':5s} range {info.range}  method {info.method}  "
          f"from {p0.explain(key)}")

Optical properties are tables against photon energy. Here is the argon's Rayleigh scattering length against
wavelength (wavelength in nm = 1239.84 / energy in eV): about a metre at argon's 128 nm scintillation light.

In [ ]:
t = p0["materials/LAr/RAYLEIGH"]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(1239.84 / np.asarray(t.energy), np.asarray(t.value))
ax.axvline(128.0, color="k", ls=":", label="128 nm")
ax.set(xlabel="wavelength (nm)", ylabel="Rayleigh scattering length (m)", yscale="log", title="liquid argon")
ax.legend()
plt.show()

And the sensors, in the order every result uses (`light.arrivals[i]` is sensor *i*), with the columns the recipe
added from sbndcode's map (each sensor's optical channel and type):

In [ ]:
cols = det.sensor_table().columns  # the id map's columns, in the order of the results
print({k: v[0] for k, v in cols.items()})
print({t: int(np.sum(np.asarray(cols["type"]) == t)) for t in np.unique(cols["type"])})

**Next:** 02 compares lumen with Geant4 on this detector.